|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 4:</h2>|<h1>The Scheduler<h1>|
|<h2>Section:</h2>|<h1>Chunked prefill<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: one token budget, two kinds of work<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Write the mixed-batch scheduler.

Each step has one token budget. A sequence that decodes contributes one token.
A sequence that prefills contributes a slice of its prompt. The scheduler
never lets a step become a long prefill that stalls everybody.

This is stage 11. It is the last scheduler in the course that is pure logic.

In [ ]:
### run this cell
from dataclasses import dataclass, replace

@dataclass(eq=False)
class Request:
  arrival_step: int
  prompt_left: int      # prompt tokens that are not prefilled yet
  output_left: int      # tokens that it must still generate

# 32 short conversations arrive at the same time and start to decode.
# Then, at step 40, a user pastes 4096 tokens.
requests = [Request(0, 64, 400) for _ in range(32)] + [Request(40, 4096, 100)]
print(f'{len(requests)} requests. The big one arrives at step {requests[-1].arrival_step}')

# Exercise 1: the budget loop

Every step spends a fixed number of tokens. Decodes first, then as much of
the waiting prompts as still fits.

In [ ]:
def decode_one_token_each(decoding, budget):
  """Give each decoding request 1 token while the budget lasts.
  Remove a request that finishes. -> the tokens used."""
  used = 0
  for request in list(decoding):
    if used >= budget:
      break
    used += 1
    request.output_left -= 1
    if request.output_left == 0:
      decoding.remove(request)
  return used

def prefill_chunks(prefilling, decoding, budget):
  """Spend the budget on prompt chunks. A request whose prompt is complete
  starts to decode. -> the tokens used."""
  used = 0
  for request in list(prefilling):
    if used >= budget:
      break
    chunk = min(request.prompt_left, budget - used)
    request.prompt_left -= chunk
    used += chunk
    if request.prompt_left == 0:
      prefilling.remove(request)
      decoding.append(request)
  return used

def schedule(requests, budget, max_running=64):
  """-> an array of rows (tokens in the step, requests that decode in the step).

  One budget for each step. Decodes go first, because each one is 1 token. A
  user who waits for the next token sees a delay sooner than a user who waits
  to start.
  """
  pending = sorted((replace(request) for request in requests),
                   key=lambda request: request.arrival_step)
  waiting, prefilling, decoding = [], [], []
  steps, step = [], 0
  while pending or waiting or prefilling or decoding:
    while pending and pending[0].arrival_step <= step:
      waiting.append(pending.pop(0))
    while waiting and len(prefilling) + len(decoding) < max_running:
      prefilling.append(waiting.pop(0))
    num_decoding = len(decoding)          # the users that wait for this step
    used = decode_one_token_each(decoding, budget)
    used += prefill_chunks(prefilling, decoding, budget - used)
    steps.append((used, num_decoding))
    step += 1
    if used == 0 and not pending:
      break
  return np.array(steps)

steps = schedule(requests, budget=512)
print(f'{len(steps)} steps, mean {steps[:,0].mean():.0f} tokens, '
      f'max {steps[:,0].max()} tokens')

# Exercise 2: put a cost on it, in the correct units

A step is not proportional to the tokens in it. The notebook
`part4_chk_theStall` measured the shape. The cost is flat up to a few hundred
tokens and linear after that. The position of the bend is a property of your
card.

So do not write milliseconds. Write the cost in **units of one plain decode
step**, and make the bend a parameter. Every number below is then a ratio, and
Exercise 3 can change the machine and change nothing else.

In [ ]:
# Step cost is not a table of milliseconds. It is the roofline of Part 1
# again: a step is free up to KNEE tokens, and linear after that.
#
#     cost(num_tokens) = max(1, num_tokens / KNEE)   in plain decode steps
#
# KNEE is the only hardware number in this notebook. Measure yours in
# part4_chk_theStall: the largest token budget whose step costs the same
# as a 32-token step.
KNEE = 256

def step_cost(num_tokens, knee=KNEE):
  """The cost of a step with num_tokens tokens, in plain decode steps."""
  return max(1.0, num_tokens / knee)

def step_costs(steps, knee=KNEE):
  """-> the cost of each row that schedule() returns."""
  return np.array([step_cost(num_tokens, knee) for num_tokens in steps[:, 0]])

def token_latencies(steps, costs):
  """-> one sample for each token that a decoding request waited for.

  A p99 over STEPS is the wrong population. One very slow step in two
  hundred does not reach the p99, but each decoding user felt it.
  """
  return np.repeat(costs, steps[:, 1].astype(int))

print(f"{'budget':>7} {'steps':>7} {'total cost':>11} {'p99 ITL':>9} {'worst ITL':>11}")
for budget in (64, 128, 256, 512, 1024, 4096):
  steps = schedule(requests, budget)
  costs = step_costs(steps)
  latencies = token_latencies(steps, costs)
  print(f'{budget:>7} {len(steps):>7} {costs.sum():>10.0f}x '
        f'{np.percentile(latencies, 99):>8.2f}x {latencies.max():>10.2f}x')
print('\nAll costs are multiples of one plain decode step.')

# Exercise 3: change the machine

Everything above used one `KNEE`. Sweep it and find the best budget for each
machine. You are looking for a rule relating the two, not three numbers.

In [ ]:
budgets = [64, 128, 256, 512, 1024, 2048, 4096]
# Sweep the machine, not the workload. For each KNEE, which budget is best?
print(f"{'KNEE':>6} " + ' '.join(f'{budget:>7}' for budget in budgets))
for knee in (64, 256, 1024):
  total_costs = []
  for budget in budgets:
    total_costs.append(step_costs(schedule(requests, budget), knee).sum())
  best = budgets[int(np.argmin(total_costs))]
  print(f'{knee:>6} ' + ' '.join(f'{cost:>7.0f}' for cost in total_costs)
        + f'   best budget {best}')

fig, axis = plt.subplots(figsize=(7.5,4.6))
for knee, style in ((64,'^--'), (256,'o-'), (1024,'s-')):
  total_costs = np.array([step_costs(schedule(requests, budget), knee).sum()
                          for budget in budgets])
  axis.plot(budgets, total_costs / total_costs.min(), style,
            label=f'KNEE={knee}, total')
axis.set_xscale('log', base=2)
axis.set(xlabel='Token budget for each step',
         ylabel="Total cost, relative to the best on that machine")
axis.legend()
axis.grid(alpha=.3)
plt.title('The best budget follows the knee, not the clock')
plt.tight_layout()
plt.show()

### One scheduler, two customers, and a statistic that lies

Throughput and tail latency point in opposite directions. This is the honest
result. It is also the reason that the budget is a configuration value and not
a constant. A large budget finishes the work sooner, and it makes every user
who decodes wait through it.

**The best budget follows `KNEE`.** Below the knee, a step costs the same as a
plain decode step. A smaller budget then buys latency that you were not going
to feel, and it costs throughput that you were going to feel. That floor comes
from the roofline in Part 1, and not from the scheduler. It moves with the
card, and not with your traffic.

### Now look at the largest budget again

Its **p99 is one of the best on the table**. Its **worst ITL is by far the
worst**. Both numbers are correct.

At budget 4096 the whole prompt goes through in two steps. Those two steps are
terrible, and every user who decodes sits through them. But two steps out of
four hundred, with thirty-two users in each step, is half of one percent of
all the token-waits in the run. It does not reach the 99th percentile. It does
not come near it.

The simulation does not cause this. It happens on real dashboards. A rare
stall that every user sees sits quietly below the p99 line. The graph stays
green, and the complaints arrive.

A percentile is a statement about a population. The population here is
token-waits. It is not users, and it is not incidents.

Stage 16 is where you choose which numbers to export. Export the maximum with
the percentile, and know which question each one answers.

### Two details in the code that are not arbitrary

- **The scheduler takes decodes before prefill chunks.** A user who watches
  the text stop sees the stall sooner than a user who saw nothing yet.
- **The prompt takes the remainder of the budget**, and not a fixed chunk. Nothing
  must decide in advance if this is a prefill step or a decode step. vLLM V1
  made this general. There are no such steps any more. There is only a token
  budget.

    ./vc guide 11